In [ ]:
# Work in /kaggle/temp so the dataset is not saved as output; only runs/ goes to /kaggle/working.
!mkdir -p /kaggle/temp
%cd /kaggle/temp

In [ ]:
%%writefile model.py
# Vanilla Vision Transformer (ViT) for CIFAR-10, written in plain PyTorch.
# Default size follows the paper's main CIFAR-10 model: width 192, depth 12, 3 heads, patch size 2.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def sinusoidal_position_embedding(num_tokens, dim):
    # Fixed (not learned) position embeddings from "Attention Is All You Need".
    # Row i is the code for position i. Even columns use sin, odd columns use cos,
    # each pair at a different frequency, so nearby positions get similar codes.
    position = torch.arange(num_tokens).unsqueeze(1)                          # (num_tokens, 1)
    freq = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))   # (dim/2,)
    pe = torch.zeros(num_tokens, dim)
    pe[:, 0::2] = torch.sin(position * freq)
    pe[:, 1::2] = torch.cos(position * freq)
    return pe                                                                 # (num_tokens, dim)


class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        # Separate layers for query, key and value so mimetic init can set each one.
        # nn.Linear stores weight as (out, in) and computes x @ weight.T + bias,
        # so the paper's W_Q (dim x dim, used as x @ W_Q) is q.weight.T here.
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)   # the paper's W_proj: mixes the heads back together

    def forward(self, x):
        B, N, D = x.shape                 # batch, tokens, embedding dim
        # Project, then split the last dim into heads: (B, N, D) -> (B, heads, N, head_dim)
        q = self.q(x).view(B, N, self.heads, self.head_dim).transpose(1, 2)
        k = self.k(x).view(B, N, self.heads, self.head_dim).transpose(1, 2)
        v = self.v(x).view(B, N, self.heads, self.head_dim).transpose(1, 2)
        # Attention: out = softmax(q @ k^T / sqrt(head_dim)) @ v, giving (B, heads, N, head_dim).
        # Each row of the softmax says how much one token looks at every other token.
        # PyTorch's built-in function computes exactly this formula, but in blocks, without
        # storing the full (B, heads, N, N) matrix. Written out by hand, that matrix needs
        # about 0.6 GB per layer at batch 512 and the T4 runs out of memory.
        out = F.scaled_dot_product_attention(q, k, v)
        # Join the heads again: (B, heads, N, head_dim) -> (B, N, D)
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.proj(out)


class Block(nn.Module):
    # One transformer layer, pre-norm: LayerNorm comes before attention and before the MLP.
    def __init__(self, dim, heads, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))  # residual connection around attention
        x = x + self.mlp(self.norm2(x))   # residual connection around the MLP
        return x


class ViT(nn.Module):
    def __init__(self, image_size=32, patch_size=2, dim=192, depth=12, heads=3, num_classes=10, pos_scale=1.0):
        super().__init__()
        num_patches = (image_size // patch_size) ** 2      # 32/2 = 16, so 16 * 16 = 256 patches
        # Cut the image into patch_size x patch_size squares and map each to a dim-vector.
        # A conv with kernel = stride = patch_size does exactly this in one step.
        self.patch_embed = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        # Extra learned token; its final state is used to classify the image.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        # Fixed sinusoidal position embeddings for the class token + all patches.
        # A buffer is saved with the model and moved to the GPU, but not trained.
        # pos_scale is gamma in Fig. 4 of the paper: a larger value makes position matter more.
        self.register_buffer("pos_embed", pos_scale * sinusoidal_position_embedding(num_patches + 1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, img):
        x = self.patch_embed(img)              # (B, 3, 32, 32) -> (B, dim, 16, 16)
        x = x.flatten(2).transpose(1, 2)       # (B, dim, 256) -> (B, 256, dim)
        cls = self.cls_token.expand(x.shape[0], -1, -1)   # one copy per image: (B, 1, dim)
        x = torch.cat([cls, x], dim=1)         # (B, 257, dim)
        x = x + self.pos_embed                 # add position codes
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])              # classify from the class token: (B, num_classes)


if __name__ == "__main__":
    # Quick check: run on Colab with `python model.py`.
    model = ViT()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"parameters: {n_params / 1e6:.2f}M")
    out = model(torch.randn(2, 3, 32, 32))
    print("output shape:", tuple(out.shape))  # expect (2, 10)
    # Check that the built-in attention equals the formula written out by hand.
    q, k, v = torch.randn(3, 2, 3, 257, 64)   # (batch, heads, tokens, head_dim) each
    manual = (q @ k.transpose(-2, -1) / math.sqrt(64)).softmax(dim=-1) @ v
    diff = (F.scaled_dot_product_attention(q, k, v) - manual).abs().max().item()
    print(f"max difference to hand-written attention: {diff:.1e}")  # expect about 1e-6 or less
    # Check pos_scale: the position embeddings should be exactly twice as large.
    ratio = (ViT(pos_scale=2.0).pos_embed / model.pos_embed)[1:]   # skip row 0: sin(0) = 0 gives 0/0
    print("pos_scale 2 ratio:", ratio.min().item(), ratio.max().item())  # expect 2.0 2.0


In [ ]:
%%writefile mimetic_init.py
# Mimetic initialization (Trockman and Kolter, ICML 2023), plain PyTorch.
# Sets the attention weights so that, from step 0:
#   W_Q W_K^T  is about  alpha1 * Z1 + beta1 * I   (per head; tokens attend to themselves)
#   W_V W_proj is about  alpha2 * Z2 - beta2 * I   (whole layer; negative diagonal)
# Z1, Z2 have entries drawn from N(0, 1/d). We follow equations 1 and 2 of the paper,
# not equations 3 to 7, which mix up the subscripts (see the report, section 1).
#
# Reminder: nn.Linear stores weight as (out, in) and computes x @ weight.T,
# so the paper's W_Q is q.weight.T, and setting W_Q = A means q.weight = A.T.

import torch
from model import Attention, ViT


@torch.no_grad()   # we are writing weights by hand, so no gradients are needed
def mimetic_init_(model, alpha1=0.7, beta1=0.7, alpha2=0.4, beta2=0.4):
    for m in model.modules():
        if not isinstance(m, Attention):
            continue
        d = m.q.weight.shape[1]   # embedding dim, 192
        k = m.head_dim            # dim per head, 64
        I = torch.eye(d)

        # Query-key: a new random target M1 for every head.
        for h in range(m.heads):
            rows = slice(h * k, (h + 1) * k)   # the output units that belong to head h
            M1 = alpha1 * torch.randn(d, d) / d ** 0.5 + beta1 * I
            U, S, Vh = torch.linalg.svd(M1)    # M1 = U diag(S) Vh, with Vh = V^T
            # Each head has only k columns, so keep the top k singular values.
            # W_Q = U[:, :k] sqrt(S), W_K = V[:, :k] sqrt(S), so W_Q W_K^T = best rank-k copy of M1.
            s = S[:k].sqrt()
            m.q.weight[rows] = (U[:, :k] * s).T   # (k, d)
            m.k.weight[rows] = s[:, None] * Vh[:k]  # (k, d), this is (V[:, :k] * s).T

        # Value-projection: one target M2 for the whole layer, full rank.
        M2 = alpha2 * torch.randn(d, d) / d ** 0.5 - beta2 * I
        U, S, Vh = torch.linalg.svd(M2)
        s = S.sqrt()
        # W_V = U sqrt(S), W_proj = sqrt(S) V^T, so W_V W_proj = M2 exactly.
        m.v.weight.copy_((U * s).T)
        m.proj.weight.copy_((s[:, None] * Vh).T)
    return model


if __name__ == "__main__":
    # Check on Colab with `python mimetic_init.py`.
    torch.manual_seed(0)
    model = mimetic_init_(ViT())
    attn = model.blocks[0].attn
    k = attn.head_dim

    def diag_report(name, P):
        d = P.shape[0]
        diag = P.diagonal().mean().item()
        off = (P - torch.diag(P.diagonal())).abs().sum().item() / (d * d - d)
        print(f"{name}: diagonal mean {diag:+.3f}, off-diagonal mean |x| {off:.3f}")

    for h in range(attn.heads):
        rows = slice(h * k, (h + 1) * k)
        WQ = attn.q.weight[rows].T             # (d, k)
        WK = attn.k.weight[rows].T             # (d, k)
        diag_report(f"head {h} W_Q W_K^T ", WQ @ WK.T)   # expect clearly positive diagonal
    WV = attn.v.weight.T
    WP = attn.proj.weight.T
    diag_report("layer   W_V W_proj", WV @ WP)            # expect about -0.4 on the diagonal


In [ ]:
%%writefile train.py
# Train a ViT on CIFAR-10, with the default init (base) or mimetic init.
# Settings follow Section 5.1 of the paper: AdamW, lr 3e-3, weight decay 0.01,
# batch 512, 100 epochs, RandAugment + Cutout. Unstated parts are our choices
# (see Training settings in the README): 5 warmup epochs then cosine decay, RandAugment(2, 9), Cutout 16x16.
#
# Example (Colab):
#   python train.py --init mimetic --out /content/drive/MyDrive/ct468/init_192
# A checkpoint is saved every epoch. Running the same command again resumes from it.

import argparse
import math
import os
import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

from model import ViT
from mimetic_init import mimetic_init_

MEAN = (0.4914, 0.4822, 0.4465)   # CIFAR-10 per-channel mean and std of the training set
STD = (0.2470, 0.2435, 0.2616)


class Cutout:
    # Set one random size x size square of the image to zero (DeVries and Taylor, 2017).
    # The center can be near the edge, so part of the square may fall outside the image.
    def __init__(self, size=16):
        self.size = size

    def __call__(self, img):                     # img: tensor (3, H, W)
        _, H, W = img.shape
        cy = torch.randint(H, (1,)).item()
        cx = torch.randint(W, (1,)).item()
        y0, y1 = max(cy - self.size // 2, 0), min(cy + self.size // 2, H)
        x0, x1 = max(cx - self.size // 2, 0), min(cx + self.size // 2, W)
        img = img.clone()
        img[:, y0:y1, x0:x1] = 0
        return img


def get_loaders(batch_size, data_dir="./data", workers=2):
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),             # shift the image by up to 4 pixels
        T.RandomHorizontalFlip(),
        T.RandAugment(num_ops=2, magnitude=9),   # 2 random ops (rotate, contrast, ...) at strength 9
        T.ToTensor(),                            # PIL image -> float tensor in [0, 1], shape (3, 32, 32)
        T.Normalize(MEAN, STD),
        Cutout(16),
    ])
    test_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])
    train_set = torchvision.datasets.CIFAR10(data_dir, train=True, download=True, transform=train_tf)
    test_set = torchvision.datasets.CIFAR10(data_dir, train=False, download=True, transform=test_tf)
    # drop_last keeps every training step the same size; the test set uses every image.
    train_loader = torch.utils.data.DataLoader(train_set, batch_size, shuffle=True,
                                               num_workers=workers, drop_last=True, pin_memory=True)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size, shuffle=False,
                                              num_workers=workers, pin_memory=True)
    return train_loader, test_loader


def lr_at(step, total_steps, warmup_steps, base_lr):
    # Linear warmup from 0 to base_lr, then cosine decay from base_lr to 0.
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)   # goes 0 -> 1
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_one_epoch(model, loader, optimizer, scaler, step, total_steps, warmup_steps, base_lr, device):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        # Set the learning rate by hand for this step.
        for group in optimizer.param_groups:
            group["lr"] = lr_at(step, total_steps, warmup_steps, base_lr)
        # Mixed precision: the forward pass runs in float16 on the GPU, which is faster.
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=scaler.is_enabled()):
            logits = model(x)
            loss = loss_fn(logits, y)
        optimizer.zero_grad(set_to_none=True)
        # The scaler multiplies the loss before backward so small float16 gradients do not become 0,
        # then divides them back before the optimizer step.
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        step += 1
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        seen += len(y)
    return total_loss / seen, 100 * correct / seen, step


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, seen = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(x)
        correct += (logits.argmax(1) == y).sum().item()
        seen += len(y)
    return 100 * correct / seen


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--init", choices=["base", "mimetic"], required=True)
    p.add_argument("--out", required=True, help="folder for checkpoint and log, e.g. on Google Drive")
    p.add_argument("--dim", type=int, default=192)
    p.add_argument("--depth", type=int, default=12)
    p.add_argument("--heads", type=int, default=3)
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--warmup", type=int, default=5, help="warmup epochs")
    p.add_argument("--batch", type=int, default=512)
    p.add_argument("--lr", type=float, default=3e-3)
    p.add_argument("--wd", type=float, default=0.01)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--pos-scale", type=float, default=1.0, help="multiply position embeddings (gamma)")
    args = p.parse_args()

    os.makedirs(args.out, exist_ok=True)
    ckpt_path = os.path.join(args.out, "checkpoint.pt")
    log_path = os.path.join(args.out, "log.csv")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(args.seed)

    train_loader, test_loader = get_loaders(args.batch)
    model = ViT(dim=args.dim, depth=args.depth, heads=args.heads, pos_scale=args.pos_scale)
    if args.init == "mimetic":
        mimetic_init_(model)          # base keeps the PyTorch default init
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

    steps_per_epoch = len(train_loader)
    total_steps = args.epochs * steps_per_epoch
    warmup_steps = args.warmup * steps_per_epoch

    # Resume if a checkpoint from an earlier (disconnected) session exists.
    start_epoch = 0
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scaler.load_state_dict(ckpt["scaler"])
        torch.set_rng_state(ckpt["rng"].cpu())   # map_location moved it to the GPU; it must be on the CPU
        start_epoch = ckpt["epoch"] + 1
        print(f"resumed from epoch {start_epoch}")
    else:
        with open(log_path, "w") as f:
            f.write("epoch,lr,train_loss,train_acc,test_acc,seconds\n")

    step = start_epoch * steps_per_epoch
    for epoch in range(start_epoch, args.epochs):
        t0 = time.time()
        train_loss, train_acc, step = train_one_epoch(model, train_loader, optimizer, scaler, step,
                                                      total_steps, warmup_steps, args.lr, device)
        test_acc = evaluate(model, test_loader, device)
        seconds = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]
        gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0
        print(f"epoch {epoch + 1}/{args.epochs}  lr {lr:.2e}  loss {train_loss:.3f}  "
              f"train {train_acc:.2f}  test {test_acc:.2f}  {seconds:.0f}s  peak GPU {gpu_gb:.1f} GB")
        with open(log_path, "a") as f:
            f.write(f"{epoch + 1},{lr:.6f},{train_loss:.4f},{train_acc:.2f},{test_acc:.2f},{seconds:.1f}\n")
        # Write to a temporary file first, so a disconnect during saving cannot corrupt the checkpoint.
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "rng": torch.get_rng_state(), "epoch": epoch,
                    "args": vars(args)}, ckpt_path + ".tmp")
        os.replace(ckpt_path + ".tmp", ckpt_path)

    print(f"done, results in {log_path}")


if __name__ == "__main__":
    main()


In [ ]:
# Check the GPU and run the self-tests.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python model.py
!python mimetic_init.py

In [ ]:
# Full runs: 100 epochs each. Logs and checkpoints end up in the notebook's Output tab.
!python train.py --init mimetic --out /kaggle/working/runs/mimetic
!python train.py --init base --out /kaggle/working/runs/base